# Scikit-Learn — Zero to Hero Worksheet
Read `concept_notes.md` and `diagrams.md` first. The pre-filled dataset in this worksheet is a custom-built version of a topic-labelled embedding space — same structure as the ChromaDB practice dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs
np.random.seed(42)
print("All imports OK")

## 1. The universal sklearn API — fit, transform, predict

In [ ]:
# Generate a small dataset to illustrate the API
X_raw = np.array([[100, 0.01],
                   [200, 0.02],
                   [ 50, 0.005],
                   [300, 0.03]])
print("Raw data (very different scales):\n", X_raw)
print("Column 0 range:", X_raw[:, 0].min(), "to", X_raw[:, 0].max())
print("Column 1 range:", X_raw[:, 1].min(), "to", X_raw[:, 1].max())

# StandardScaler: subtract mean, divide by std — puts all columns on same scale
scaler = StandardScaler()      # Step 1: create
scaler.fit(X_raw)              # Step 2: learn (calculates mean and std per column)
X_scaled = scaler.transform(X_raw)   # Step 3: apply

print("\nScaled data:\n", X_scaled.round(3))
print("Column 0 after scaling — mean:", X_scaled[:, 0].mean().round(6),
      " std:", X_scaled[:, 0].std().round(3))
print("Column 1 after scaling — mean:", X_scaled[:, 1].mean().round(6),
      " std:", X_scaled[:, 1].std().round(3))

# Or combined into one call
X_scaled2 = StandardScaler().fit_transform(X_raw)
print("\nfit_transform result identical:", np.allclose(X_scaled, X_scaled2))

### Observe:
- After scaling, both columns have mean ~0 and std ~1. Why is this important before running KMeans? (Hint: KMeans uses Euclidean distance — what happens if one column has values 0-300 and another has 0-0.03?)
- Try `scaler.mean_` and `scaler.scale_` — what are these?

## 2. PCA — reducing dimensions for visualization

In [ ]:
# Build a synthetic 50-dimensional dataset with 3 true clusters
# (like embeddings with only 3 topics)
X, y_true = make_blobs(n_samples=90, centers=3, n_features=50,
                         cluster_std=2.0, random_state=42)
print("Original shape:", X.shape, "— 90 documents, 50 features each")

# Reduce to 2D
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
print("After PCA:", X_2d.shape, "— 90 documents, 2 features each")

# How much information did we keep?
print("Variance explained by component 1:", f"{pca.explained_variance_ratio_[0]:.1%}")
print("Variance explained by component 2:", f"{pca.explained_variance_ratio_[1]:.1%}")
print("Total variance kept (2 components):", f"{pca.explained_variance_ratio_.sum():.1%}")

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#E74C3C", "#3498DB", "#2ECC71"]
for i in range(3):
    idx = y_true == i
    ax.scatter(X_2d[idx, 0], X_2d[idx, 1],
                color=colors[i], label=f"Cluster {i}", alpha=0.7, s=50)
ax.set_title("PCA: 50D → 2D (colored by true cluster)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.legend()
plt.show()

### Observe:
- Are the 3 clusters cleanly separated in 2D? Does the percentage of variance explained help explain how well-separated they look?
- Try `n_components=50` and check `pca.explained_variance_ratio_.sum()` — what's the value now and why?
- Try `n_components=10` — is the visual separation in the 2D plot any better?

## 3. PCA vs t-SNE — when they give different answers

In [ ]:
# Build a harder dataset: 8 tight clusters (like your 8-topic embedding data)
X8, y8 = make_blobs(n_samples=120, centers=8, n_features=100,
                      cluster_std=1.5, random_state=42)

# PCA
pca8 = PCA(n_components=2)
X_pca = pca8.fit_transform(X8)

# t-SNE (slower — perplexity should be smaller than n_samples/3 as rough rule)
tsne = TSNE(n_components=2, perplexity=15, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.get_cmap("tab10")

for i in range(8):
    idx = y8 == i
    axes[0].scatter(X_pca[idx, 0], X_pca[idx, 1],
                     color=cmap(i), label=f"T{i}", alpha=0.7, s=40)
    axes[1].scatter(X_tsne[idx, 0], X_tsne[idx, 1],
                     color=cmap(i), label=f"T{i}", alpha=0.7, s=40)

axes[0].set_title(f"PCA (variance kept: {pca8.explained_variance_ratio_.sum():.1%})")
axes[1].set_title("t-SNE (perplexity=15)")
for ax in axes:
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

print("PCA variance by component:", pca8.explained_variance_ratio_.round(3))

### Observe:
- Which method separates the 8 clusters more cleanly in this case?
- t-SNE is slower. Try timing both: `%%timeit` at the top of a cell or `import time; t0=time.time(); ...; print(time.time()-t0)`.
- Try changing `perplexity` in t-SNE to 5, then 50. What changes?

## 4. KMeans — finding clusters without labels
This is the most-used section for the ChromaDB lab.

In [ ]:
# We'll use the 8-cluster dataset but PRETEND we don't know the labels
# i.e., we're discovering structure in unsupervised fashion

# Fit KMeans
km = KMeans(n_clusters=8, random_state=42, n_init=10)
km.fit(X8)
predicted_labels = km.labels_

# Silhouette score — how well-separated are the clusters?
score = silhouette_score(X8, predicted_labels)
print(f"Silhouette score (k=8): {score:.3f}")
print("(Range: -1 bad → 0 random → 1 perfect)")

# Visualize predicted clusters vs true clusters
X_pca_show = PCA(n_components=2).fit_transform(X8)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.get_cmap("tab10")

for i in range(8):
    idx_true = y8 == i
    idx_pred = predicted_labels == i
    axes[0].scatter(X_pca_show[idx_true, 0], X_pca_show[idx_true, 1],
                     color=cmap(i), s=40, alpha=0.7)
    axes[1].scatter(X_pca_show[idx_pred, 0], X_pca_show[idx_pred, 1],
                     color=cmap(i), s=40, alpha=0.7)

axes[0].set_title("True clusters (ground truth labels)")
axes[1].set_title(f"KMeans predicted clusters (k=8, sil={score:.3f})")
plt.tight_layout()
plt.show()

### Observe:
- Do the predicted clusters match the true clusters?
- KMeans can't guarantee the *color assignments* match (cluster 0 predicted ≠ topic 0 necessarily) — only that the groupings match. This is called the 'label permutation' problem in clustering.
- If the predicted clusters look different from the true ones, does the silhouette score still look good?

## 5. The elbow method — choosing the right K

In [ ]:
# Use the elbow method to discover K without knowing the true labels
inertias = []
silhouettes = []
k_values = range(2, 12)

for k in k_values:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X8)
    inertias.append(km_temp.inertia_)
    silhouettes.append(silhouette_score(X8, km_temp.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(k_values, inertias, marker="o", color="steelblue")
axes[0].axvline(x=8, color="red", linestyle="--", alpha=0.7, label="true k=8")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (lower = tighter clusters)")
axes[0].set_title("Elbow Method — look for the bend")
axes[0].legend()

axes[1].plot(k_values, silhouettes, marker="o", color="green")
axes[1].axvline(x=8, color="red", linestyle="--", alpha=0.7, label="true k=8")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score (higher = better)")
axes[1].set_title("Silhouette Score — look for the peak")
axes[1].legend()

plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(silhouettes)]
print(f"Best k by silhouette score: {best_k} (true k was 8)")

### Observe:
- Does the elbow in the inertia plot land at k=8, or nearby?
- Does the silhouette score peak at k=8, or somewhere else?
- These two methods sometimes disagree. When they do, silhouette score is generally more reliable — explain why using the definitions of each metric.

## 6. The teaser: PCA visual vs silhouette score can disagree

In [ ]:
# Build a deliberately tricky dataset: 3 visually-distinct PCA blobs
# but really 8 sub-clusters hidden inside each blob
from sklearn.datasets import make_blobs

# 8 tight hidden sub-clusters, but arranged so PCA only shows 3 blobs
centers_8 = [
    [-5, -5], [-4.8, -4.6], [-5.2, -4.8],   # 3 sub-clusters inside "blob A"
    [ 0,  0], [ 0.2,  0.1],                  # 2 sub-clusters inside "blob B"
    [ 5,  5], [ 5.1, 4.9], [ 4.8, 5.2],     # 3 sub-clusters inside "blob C"
]
X_tricky = []
y_tricky_8 = []   # true 8-cluster labels
y_tricky_3 = []   # apparent 3-blob labels

blob_membership = [0,0,0, 1,1, 2,2,2]
for i, center in enumerate(centers_8):
    pts = np.random.randn(15, 50) * 0.3 + np.array(center + [0]*48)
    X_tricky.append(pts)
    y_tricky_8.extend([i]*15)
    y_tricky_3.extend([blob_membership[i]]*15)

X_tricky = np.vstack(X_tricky)
y_tricky_8 = np.array(y_tricky_8)
y_tricky_3 = np.array(y_tricky_3)

coords_tricky = PCA(n_components=2).fit_transform(X_tricky)
score_k3 = silhouette_score(X_tricky, KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_tricky))
score_k8 = silhouette_score(X_tricky, KMeans(n_clusters=8, n_init=10, random_state=42).fit_predict(X_tricky))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap3 = plt.get_cmap("Set1")
cmap8 = plt.get_cmap("tab10")

for i in range(3):
    idx = y_tricky_3 == i
    axes[0].scatter(coords_tricky[idx, 0], coords_tricky[idx, 1],
                     color=cmap3(i), s=40, alpha=0.8)
axes[0].set_title(f"PCA shows 3 blobs → k=3 feels right
silhouette(k=3)={score_k3:.3f}")

for i in range(8):
    idx = y_tricky_8 == i
    axes[1].scatter(coords_tricky[idx, 0], coords_tricky[idx, 1],
                     color=cmap8(i), s=40, alpha=0.8)
axes[1].set_title(f"But k=8 is the true structure
silhouette(k=8)={score_k8:.3f}")

plt.suptitle("The teaser: visual intuition vs numerical measurement", y=1.02)
plt.tight_layout()
plt.show()

print(f"Silhouette with k=3: {score_k3:.3f}")
print(f"Silhouette with k=8: {score_k8:.3f}")
print(f"→ k=8 scores {'better' if score_k8 > score_k3 else 'worse'} despite looking the same in 2D PCA")

### Final exercise — apply everything to the real ChromaDB practice dataset
```python
import pandas as pd
df = pd.read_excel('chroma_practice_lab/dataset/practice_dataset.xlsx')
# Get real embedding vectors via InHouseEmbeddings()
# (requires the chroma_lab_env venv and inhouse_llm.py)
```
Then:
1. Run PCA to 2D, color by `topic` — do 8 clusters appear visually?
2. Run the elbow method from k=2 to k=12 on the REAL embeddings — does silhouette score peak at k=8?
3. Compare t-SNE and PCA side by side on the real data.
4. Compute `silhouette_score` for k=3 and k=8 on the real data — does the same pattern from the teaser above show up?

**This exercise ties together everything in this module with the actual data you've already been working with.**